# UA-FCCL: concept counts and ConExp contexts for all nine datasets (Colab)
Run the cells **one by one, from top to bottom**. If this Colab session was used before, first click **Runtime > Restart session**.

**Files needed on your computer:** `Shill Bidding Dataset.csv`. and `lattice_count.py` (sent in chat). If `lattice_count.py` is already in the GitHub repository, Cell 3b finds it by itself.

For every dataset and for θ = 0.3, 0.5 and 0.7 (split 0), the notebook writes the context as a `.cxt` file for ConExp and counts the formal concepts. `concepts_with_bottom` is the number ConExp shows. `concepts_without_empty_bottom` is the older count, which leaves out the bottom concept when no training object has all attributes.

## Cell 1: get the repo and one package

In [ ]:
%cd /content
!rm -rf conformal-fuzzy-ccl
!git clone -q https://github.com/mahibgm9676-lab/conformal-fuzzy-ccl.git
%cd /content/conformal-fuzzy-ccl
!pip -q install ucimlrepo
import os
os.environ['CCL_DING_DIR'] = '/content/class-conditional-conformal'
!ls ccl_helpers.py lattice_count.py

## Cell 2: download the datasets (ignore the Shill Bidding message)

In [ ]:
!python 01_prepare_data.py
!ls data

## Cell 3: upload `Shill Bidding Dataset.csv`
This cell also creates the `data` folder if it is missing, and it works when Colab renames the file to `... (1).csv`.

In [ ]:
import glob, shutil, os
from google.colab import files
os.makedirs('data', exist_ok=True)
up = files.upload()
c = sorted(glob.glob('Shill Bidding Dataset*.csv'), key=os.path.getmtime)
shutil.copy(c[-1], 'data/Shill Bidding Dataset.csv')
print('using', c[-1])
!ls data

## Cell 3b: make sure `lattice_count.py` is here
If the file is not in the GitHub repository yet, this cell asks you to choose it from your computer. It prints `True` when the file is found.

In [ ]:
import glob, shutil, os
if not os.path.exists('lattice_count.py'):
    from google.colab import files
    up = files.upload()
    c = sorted(glob.glob('lattice_count*.py'), key=os.path.getmtime)
    if c[-1] != 'lattice_count.py':
        shutil.copy(c[-1], 'lattice_count.py')
print(os.path.exists('lattice_count.py'))

## Cell 4: write the context files and count the concepts (all nine datasets, three thresholds)
Dry Bean has 16 attributes. It takes a few seconds per threshold.

In [ ]:
import time, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
from lattice_count import export_cxt_all
files_map = {'shill': 'Shill Bidding Dataset.csv', 'wilt': 'wilt.csv', 'htru2': 'htru2.csv', 'magic': 'magic.csv',
             'pageblocks': 'pageblocks.csv', 'drybean': 'drybean.csv', 'shuttle': 'shuttle.csv',
             'yeast': 'yeast.csv', 'ecoli': 'ecoli.csv'}
os.makedirs('cxt_all', exist_ok=True)
rows = []
for n, f in files_map.items():
    for th in [0.3, 0.5, 0.7]:
        t0 = time.time()
        g, a, w, wo = export_cxt_all('data/' + f, th, 0, f'cxt_all/{n}_theta{th}.cxt')
        rows.append([n, th, g, a, w, wo])
        print(n, th, 'groups', g, 'attributes', a, 'concepts', w, wo, round(time.time() - t0, 1), 's')
T = pd.DataFrame(rows, columns=['dataset', 'theta', 'object_groups', 'attributes', 'concepts_with_bottom', 'concepts_without_empty_bottom'])
T.to_csv('cxt_all/concept_counts_all.csv', index=False)
print(T.to_string(index=False))

## Cell 5: download the files and send me the counts table

In [ ]:
!zip -q cxt_all_files.zip cxt_all/*
from google.colab import files
files.download('cxt_all_files.zip')